In [1]:
import sys
import os

sys.path.append(os.path.abspath("../src"))

import pandas as pd
import numpy as np
from utils import audit, load_and_split_city, diagnose_missing, impute_time_series

# 1. Load dữ liệu cho Delhi
delhi = load_and_split_city("Delhi")
print("=== DELHI (audit gốc) ===")
display(audit(delhi).sort_values("pct_missing", ascending=False))

# 2. Load dữ liệu cho Gurugram
gurugram = load_and_split_city("Gurugram")
print("\n=== GURUGRAM (audit gốc) ===")
display(audit(gurugram).sort_values("pct_missing", ascending=False))


=== DELHI (audit gốc) ===


,dtype,n_missing,pct_missing,n_unique
Xylene,float64,781,38.9,387
SO2,float64,110,5.5,1284
O3,float64,84,4.2,1684
PM10,float64,77,3.8,1888
AQI,float64,10,0.5,471
AQI_Bucket,str,10,0.5,6
NH3,float64,9,0.4,1604
NO,float64,2,0.1,1745
PM2.5,float64,2,0.1,1913
NO2,float64,2,0.1,1764



=== GURUGRAM (audit gốc) ===


,dtype,n_missing,pct_missing,n_unique
NH3,float64,1560,92.9,119
Xylene,float64,1560,92.9,104
PM10,float64,939,55.9,732
Toluene,float64,610,36.3,390
NOx,float64,452,26.9,1017
Benzene,float64,442,26.3,350
AQI,float64,226,13.5,421
AQI_Bucket,str,226,13.5,6
SO2,float64,175,10.4,1024
PM2.5,float64,154,9.2,1461


In [2]:
# Chẩn đoán: cột nào nên nội suy trực tiếp, cột nào có khối lớn cần cẩn thận, cột nào nên đề xuất DROP
print("=== DELHI - diagnose_missing ===")
display(diagnose_missing(delhi))

print("\n=== GURUGRAM - diagnose_missing ===")
display(diagnose_missing(gurugram))


=== DELHI - diagnose_missing ===


,pct_missing,block_ratio_%,action
col,,,
PM2.5,0.1,100.0,nội suy trực tiếp (missing quá ít)
PM10,3.8,75.0,"có khối lớn — chỉ nội suy phần rải rác, giữ Na..."
NO,0.1,100.0,nội suy trực tiếp (missing quá ít)
NO2,0.1,100.0,nội suy trực tiếp (missing quá ít)
NOx,0.0,NaN,giữ nguyên
NH3,0.4,22.0,nội suy trực tiếp (missing quá ít)
CO,0.0,NaN,giữ nguyên
SO2,5.5,74.0,"có khối lớn — chỉ nội suy phần rải rác, giữ Na..."
O3,4.2,96.0,"có khối lớn — chỉ nội suy phần rải rác, giữ Na..."



=== GURUGRAM - diagnose_missing ===


,pct_missing,block_ratio_%,action
col,,,
PM2.5,9.2,36,nội suy trực tiếp
PM10,55.9,81,"có khối lớn — chỉ nội suy phần rải rác, giữ Na..."
NO,8.2,38,nội suy trực tiếp
NO2,8.1,38,nội suy trực tiếp
NOx,26.9,72,"có khối lớn — chỉ nội suy phần rải rác, giữ Na..."
NH3,92.9,100,ĐỀ XUẤT DROP (gần như không có dữ liệu thật)
CO,6.5,41,nội suy trực tiếp
SO2,10.4,17,nội suy trực tiếp
O3,6.9,32,nội suy trực tiếp


In [3]:
delhi_clean, delhi_flat_changed = impute_time_series(delhi, "Delhi", drop_cols=[])
print()
gurugram_clean, gurugram_flat_changed = impute_time_series(
    gurugram, "Gurugram", drop_cols=["NH3", "Xylene"]
)


--- Xử lý Missing Values cho Delhi ---
  - CO: phát hiện 84 ngày flatline (0 liên tục >= 3 ngày) -> NaN
  - Benzene: phát hiện 3 ngày flatline (0 liên tục >= 3 ngày) -> NaN
  - Toluene: phát hiện 105 ngày flatline (0 liên tục >= 3 ngày) -> NaN
  - Xylene: phát hiện 128 ngày flatline (0 liên tục >= 3 ngày) -> NaN
  => Còn 1161 ô missing SAU xử lý (CHỦ Ý giữ NaN ở khối trống dài, không bịa số)

--- Xử lý Missing Values cho Gurugram ---
  - NOx: phát hiện 23 ngày flatline (0 liên tục >= 3 ngày) -> NaN
  - CO: phát hiện 30 ngày flatline (0 liên tục >= 3 ngày) -> NaN
  - Benzene: phát hiện 264 ngày flatline (0 liên tục >= 3 ngày) -> NaN
  - Toluene: phát hiện 253 ngày flatline (0 liên tục >= 3 ngày) -> NaN
  - Đã xoá cột (đề xuất, cần nhóm đồng ý): ['NH3', 'Xylene']
  => Còn 3043 ô missing SAU xử lý (CHỦ Ý giữ NaN ở khối trống dài, không bịa số)


In [4]:
# Kiểm chứng: so sánh % missing trước / sau cho từng cột
pollutant_cols = ['PM2.5','PM10','NO','NO2','NOx','NH3','CO','SO2','O3','Benzene','Toluene','Xylene']

def compare_missing(before_df, after_df, cols):
    before = before_df[cols].isna().mean() * 100
    after_cols = [c for c in cols if c in after_df.columns]
    after = after_df[after_cols].isna().mean() * 100
    return pd.DataFrame({'truoc_%': before.round(1), 'sau_%': after.reindex(before.index).round(1)})

print("=== DELHI: truoc/sau ===")
display(compare_missing(delhi, delhi_clean, pollutant_cols + ['AQI']))

print("\n=== GURUGRAM: truoc/sau ===")
display(compare_missing(gurugram, gurugram_clean, pollutant_cols + ['AQI']))


=== DELHI: truoc/sau ===


,truoc_%,sau_%
PM2.5,0.1,0.0
PM10,3.8,2.2
NO,0.1,0.0
NO2,0.1,0.0
NOx,0.0,0.0
NH3,0.4,0.0
CO,0.0,3.5
SO2,5.5,4.0
O3,4.2,3.3
Benzene,0.0,0.0



=== GURUGRAM: truoc/sau ===


,truoc_%,sau_%
PM2.5,9.2,3.8
PM10,55.9,51.5
NO,8.2,2.3
NO2,8.1,2.3
NOx,26.9,21.3
NH3,92.9,NaN
CO,6.5,1.8
SO2,10.4,1.4
O3,6.9,1.4
Benzene,26.3,35.9


In [5]:
os.makedirs("../data/interim", exist_ok=True)
delhi_clean.to_csv("../data/interim/delhi_after_missing.csv", index=False)
gurugram_clean.to_csv("../data/interim/gurugram_after_missing.csv", index=False)
print("Đã lưu data/interim/delhi_after_missing.csv và data/interim/gurugram_after_missing.csv")
print("Delhi:", delhi_clean.shape, " | Gurugram:", gurugram_clean.shape)


Đã lưu data/interim/delhi_after_missing.csv và data/interim/gurugram_after_missing.csv
Delhi: (2009, 29)  | Gurugram: (1679, 27)
